Visualização dos dados


In [ ]:
!pip install minisom

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from collections import Counter
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.cluster import KMeans, DBSCAN
from minisom import MiniSom
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

print("--- Iniciando Carregamento ---")

try:
    df = pd.read_csv('./creditcard.csv')
    print("Base de dados carregada com sucesso!")
    print(f"Dimensões: {df.shape[0]} linhas e {df.shape[1]} colunas.")

except FileNotFoundError:
  print(" Arquivo 'creditcard.csv' não encontrado.")

print("\n--- Gerando Histogramas... ---")
sns.set(style="white")

df.hist(figsize=(24, 18), bins=50, color='#5f9ea0', edgecolor='black', grid=False)

plt.suptitle("Distribuição das Variáveis (Histogramas)", fontsize=22, y=1.02)
plt.tight_layout()
plt.show()

print("\n--- Gerando Gráfico de Barras da Classe (Nominal) ---")

plt.figure(figsize=(8, 6))
ax = sns.countplot(x='Class', data=df, palette=['#1f77b4', '#d62728'])

plt.title('Distribuição da Classe (0 = Normal, 1 = Fraude)', fontsize=16)
plt.xlabel('Classe Nominal')
plt.ylabel('Contagem (Quantidade de Transações)')
plt.yscale('log')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + 0.35, p.get_height() * 1.1), fontsize=12)

plt.show()

df.info()
print(f"-> Dimensões: {df.shape}")

Seleção de atributos

In [ ]:
print("\n--- 2. Seleção de Atributos ---")

if 'Time' in df.columns:
    df = df.drop('Time', axis=1)

    print("-> Atributo 'Time' removido.")

print(f"-> Dimensões: {df.shape}")

Codificação

In [ ]:
print("\n--- 3. Codificação ---")
cols_texto = df.select_dtypes(include=['object']).columns
if len(cols_texto) > 0:
    print(f"Codificando colunas: {cols_texto}")
    df = pd.get_dummies(df, columns=cols_texto, drop_first=True)
else:
    print("Nenhuma coluna de texto encontrada. Dataset já é numérico.")

Eliminação de Inconsistência (mesmos dados e classes diferentes) e Redundância (mesmos dados)

In [ ]:
print("\n--- 4. Eliminação de Redundâncias ---")
duplicadas = df.duplicated().sum()
print(f"Linhas duplicadas encontradas: {duplicadas}")
if duplicadas > 0:
    df.drop_duplicates(inplace=True)
    print(f"-> Duplicatas removidas. Novas dimensões: {df.shape}")

Eliminação de outliers

In [ ]:
print("\n--- 5. Eliminação de Outliers (Ajustado) ---")
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
limite_sup = Q3 + 10.0 * IQR  # Aumentado para ser menos agressivo

outliers_idx = df[df['Amount'] > limite_sup].index
print(f"Outliers extremos em 'Amount' detectados: {len(outliers_idx)}")
df = df.drop(outliers_idx)
print(f"-> Outliers removidos. Novas dimensões: {df.shape}")

Matriz de Correlação

In [ ]:
print("\n--- Análise de Correlação (Pós-Limpeza) ---")
plt.figure(figsize=(24, 18))
sns.heatmap(df.corr(), cmap='coolwarm', annot=False, linewidths=0.5)
plt.title("Matriz de Correlação (Dados Reais Limpos)")
plt.show()

Árvore de Correlação

In [ ]:
print("\n--- Visualizando a Árvore de Decisão (Exploração) ---")

X_para_plot = df.drop('Class', axis=1)
y_para_plot = df['Class']

modelo_arvore = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced')
modelo_arvore.fit(X_para_plot, y_para_plot)

# Plotagem
plt.figure(figsize=(24, 10))
plot_tree(modelo_arvore,
          feature_names=X_para_plot.columns.tolist(),
          class_names=['Normal', 'Fraude'],
          filled=True,
          rounded=True,
          fontsize=12)

plt.title("Regras de Decisão Principais (Visão Geral)", fontsize=18)
plt.show()

Divisão treino e teste

In [ ]:
print("\n--- Divisão Treino-Teste ---")
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

Imputação de dados ausentes (Treino e Teste)

In [ ]:
print("\n--- 6. Imputação de Dados Ausentes ---")
imputer = SimpleImputer(strategy='median')

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

cols = X.columns
X_train = pd.DataFrame(X_train_imputed, columns=cols)
X_test = pd.DataFrame(X_test_imputed, columns=cols)
print("-> Valores nulos preenchidos com a mediana.")

Normalização e padronização (Treino e Teste)

In [ ]:
print("\n--- 7. Normalização ---")
scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled, columns=cols)
X_test = pd.DataFrame(X_test_scaled, columns=cols)
print("-> Dados normalizados com RobustScaler.")

Balanceamento (Apenas treino)

In [ ]:
print("\n--- 8. Balanceamento ---")

print(f"Contagem Original (Treino): {Counter(y_train)}")

# OVERSAMPLING
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_over, y_over = smote.fit_resample(X_train, y_train)

print(f"\n[A] Oversampling (SMOTE):")
print(f"   Resultado: {Counter(y_over)}")

# UNDERSAMPLING
rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train)

print(f"\n[B] Undersampling:")
print(f"   Resultado: {Counter(y_under)}")

#  HÍBRIDO (Over + Under)
pipeline_hibrido = Pipeline(steps=[
    ('over', SMOTE(sampling_strategy=0.008, random_state=42)),
    ('under', RandomUnderSampler(sampling_strategy=0.5, random_state=42))
])
X_hibrido, y_hibrido = pipeline_hibrido.fit_resample(X_train, y_train)

print(f"\n[C] Híbrido (Over + Under):")
print(f"   Resultado: {Counter(y_hibrido)}")

X_train_final = X_hibrido
y_train_final = y_hibrido

# X_train_final = X_over
# y_train_final = y_over

# X_train_final = X_under
# y_train_final = y_under

print(f"\n Dataset definido para treinamento: {len(X_train_final)} linhas.")


Comparação Antes vs Depois balanceamento

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

contagem_antes = Counter(y_train)
sns.barplot(x=list(contagem_antes.keys()), y=list(contagem_antes.values()), ax=ax[0], palette='Blues')
ax[0].set_title(f"Antes do Balanceamento\n(Fraudes: {contagem_antes[1]})", fontsize=14)
ax[0].set_xlabel("Classe (0=Normal, 1=Fraude)")
ax[0].set_ylabel("Quantidade")
ax[0].set_yscale("log")
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height())}', (p.get_x() + 0.4, p.get_height()), ha='center', va='bottom')

var_balanceada = y_train_final

contagem_depois = Counter(var_balanceada)
sns.barplot(x=list(contagem_depois.keys()), y=list(contagem_depois.values()), ax=ax[1], palette='Greens')
ax[1].set_title(f"Após Balanceamento Híbrido\n(Fraudes: {contagem_depois[1]})", fontsize=14)
ax[1].set_xlabel("Classe (0=Normal, 1=Fraude)")
ax[1].set_ylabel("Quantidade")

for p in ax[1].patches:
    ax[1].annotate(f'{int(p.get_height())}', (p.get_x() + 0.4, p.get_height()), ha='center', va='bottom')

plt.tight_layout()
plt.show()

Avaliação do modelo

In [ ]:
scale_pos_weight = (y_train_final == 0).sum() / (y_train_final == 1).sum()

novos_modelos = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss'
    ),
    "Regressão Logística": LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    ),
    "Gradient Boosting (Sklearn)": GradientBoostingClassifier(
        random_state=42
    )
}

print(f"--- Iniciando Bateria de Testes Ampliada ---")
print(f"Treinando com dados balanceados: {X_train_final.shape}")
print("-" * 60)

for nome, modelo in novos_modelos.items():
    print(f" Treinando {nome}...")

    try:
        modelo.fit(X_train_final, y_train_final)

        y_pred = modelo.predict(X_test)

        print(f"\n Resultados: {nome}")
        print(classification_report(y_test, y_pred))

        plt.figure(figsize=(5, 3))
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f'{nome}')
        plt.ylabel('Real')
        plt.xlabel('Previsto')
        plt.show()
        print("-" * 60)

    except Exception as e:
        print(f" Erro no {nome}: {e}")
        print("-" * 60)

Comparativo visual

In [ ]:
dados_resultados = {
    'Modelo': ['Random Forest', 'XGBoost', 'Regressão Logística', 'Gradient Boosting'],
    'Recall (Fraude)': [0.80, 0.84, 0.86, 0.84],
    'Precision (Fraude)': [0.47, 0.20, 0.06, 0.16],
    'F1-Score (Fraude)': [0.59, 0.32, 0.12, 0.28]
}

df_res = pd.DataFrame(dados_resultados)

# Plotagem
df_melted = df_res.melt(id_vars='Modelo', var_name='Métrica', value_name='Valor')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x='Modelo', y='Valor', hue='Métrica', palette='viridis')
plt.title('Comparativo de Performance na Detecção de Fraudes (Classe 1)', fontsize=16)
plt.ylim(0, 1.0)
plt.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Meta de Recall (80%)')
plt.legend(loc='upper right')
plt.show()

# Algoritmos de Clustering


In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X_cluster = df.drop('Class', axis=1).values
y_real = df['Class'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"Dados preparados para agrupamento. Formato: {X_scaled.shape}")

Plotagem

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
sample_size = 10000
np.random.seed(42)
indices = np.random.choice(X_scaled.shape[0], sample_size, replace=False)
X_sample = X_scaled[indices]
y_sample_real = y_real[indices] # Apenas para referência visual

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_sample)

print(f"Dados reduzidos de {X_scaled.shape[1]} dimensões para 2 (PCA) para visualização.")

# --- FUNÇÃO PARA PLOTAR ---
def plot_clusters(model_name, labels, X_2d):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=X_2d[:,0], y=X_2d[:,1], hue=labels, palette='viridis', s=30, alpha=0.6)
    plt.title(f"Agrupamento: {model_name} (Visualização via PCA)", fontsize=15)
    plt.xlabel("Componente Principal 1")
    plt.ylabel("Componente Principal 2")
    plt.legend(title='Cluster Encontrado')
    plt.show()

def print_metrics(model_name, X, labels):
    if len(set(labels)) > 1:
        sil = silhouette_score(X, labels)
        db = davies_bouldin_score(X, labels)
        print(f"--- Métricas {model_name} ---")
        print(f"Silhouette Score: {sil:.4f} (Melhor perto de 1)")
        print(f"Davies-Bouldin: {db:.4f}  (Melhor perto de 0)")
    else:
        print(f"--- {model_name}: Não foi possível calcular métricas (apenas 1 grupo encontrado).")

K-means

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

In [ ]:
print("\nRodando K-Means...")
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_sample)

print_metrics("K-Means", X_sample, labels_kmeans)
plot_clusters("K-Means (Tenta dividir em 2)", labels_kmeans, X_pca)

DBSCAN

In [ ]:
print("\nRodando DBSCAN...")
dbscan = DBSCAN(eps=5.0, min_samples=10)
labels_dbscan = dbscan.fit_predict(X_sample)

n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
print(f"DBSCAN encontrou {n_clusters} grupos e {list(labels_dbscan).count(-1)} pontos de ruído.")

print_metrics("DBSCAN", X_sample, labels_dbscan)
plot_clusters("DBSCAN (Cor -1 é Ruído/Fraude)", labels_dbscan, X_pca)

SOM

In [ ]:
print("\nRodando SOM (Self-Organizing Maps)...")
# Mapa 15x15
dim = 15
som = MiniSom(dim, dim, X_sample.shape[1], sigma=1.0, learning_rate=0.5)
som.random_weights_init(X_sample)
som.train_random(X_sample, num_iteration=1000)
plt.figure(figsize=(12, 10))
plt.title("SOM - Mapa de Distâncias (Pontos Vermelhos = Fraudes Reais)")
plt.pcolor(som.distance_map().T, cmap='bone_r')
plt.colorbar()

for i, x in enumerate(X_sample):
    w = som.winner(x)
    if y_sample_real[i] == 1:
        plt.plot(w[0] + 0.5, w[1] + 0.5, 'X', markeredgecolor='red',
                 markerfacecolor='None', markersize=10, markeredgewidth=2)

plt.show()